In [ ]:
import json
import time
from datetime import datetime, timezone
from dateutil import parser as dtparser

import requests
import pandas as pd

In [ ]:
GAMMA_BASE = "https://gamma-api.polymarket.com"  # market + event metadata
# (Later you may also use data feeds / websockets, but start with Gamma for ingestion)

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "bit-polymarket-signal-scanner/0.1"})

def get_json(url, params=None, timeout=30, max_retries=3, backoff=1.5):
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            sleep_s = backoff ** attempt
            print(f"[warn] GET failed ({attempt+1}/{max_retries}) -> {e}. Sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_err

In [ ]:
slug = "what-price-will-bitcoin-hit-in-february-2026"  # example slug used in docs
url = f"{GAMMA_BASE}/events"
data = get_json(url, params={"slug": slug})

type(data), (len(data) if isinstance(data, list) else data.keys())

In [ ]:
print(json.dumps(data[0], indent=2)[:10000])  # print first ~4k chars

In [ ]:
# The "Search Markets" doc page corresponds to Gamma's public search endpoint.
# We'll call it on gamma-api with a /search route.

query = "tariff"
search_url = f"{GAMMA_BASE}/public-search"
res = get_json(search_url, params={
    "q": query,
    "events_status": "active",  # try "active" first
    "limit_per_type": 20,
    "page": 1,
    "search_tags": True,
    "search_profiles": False,
})

print(res.keys())
print("markets:", len(res.get("markets", [])))
print("events:", [event["title"] for event in res.get("events", [])])
print("tags:", res.get("tags", []))

In [ ]:
def brief_market(m):
    return {
        "id": m.get("id"),
        "slug": m.get("slug"),
        "title": m.get("title") or m.get("question"),
        "closed": m.get("closed"),
        "endDate": m.get("endDate"),
        "volume": m.get("volume"),
        
    }

df = pd.DataFrame([brief_market(m) for m in res.get("markets", [])]).head(10)
df

In [ ]:
events_url = f"{GAMMA_BASE}/events"

# Start minimal; if you get too much data, add pagination/filters.
events = get_json(events_url, params={"status": "active", "limit": 20})

# events is typically a list; inspect first item
print(type(events), len(events))
print(json.dumps(events[0], indent=2)[:3500])

In [ ]:
markets_url = f"{GAMMA_BASE}/markets"

# pick 1 event
event_id = events[0].get("id")
event_slug = events[0].get("slug")
print("event_id:", event_id, "event_slug:", event_slug)

markets_for_event = get_json(markets_url, params={"event_id": event_id})
print("markets_for_event:", len(markets_for_event))
pd.DataFrame([brief_market(m) for m in markets_for_event]).head(10)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env", override=True)

from polyscanner.llm_gemini import generate_json

print("Has GOOGLE_API_KEY?", bool(os.getenv("GOOGLE_API_KEY")))
print("Model:", os.getenv("GEMINI_MODEL"))

out = generate_json(
    prompt='{"ping":"hello","instructions":"Return ONLY a JSON object with key ok=true"}',
    system="Return only JSON.",
    temperature=0.0,
)
out


In [ ]:
from polyscanner.pipeline.minimal import run_minimal_pipeline

result = run_minimal_pipeline(top_n=10)  # uses DATABASE_URL + GOOGLE_API_KEY from .env
result["report_path"], result["llm_result"]
result["report_md"] #contains the full markdown text


import pandas as pd

items = result["llm_result"]["items"]
df = pd.DataFrame(items)

# optional: join market text back in (from ranked_markets)
markets = pd.DataFrame(result["ranked_markets"])[["pm_market_id", "question", "probability", "volume", "score"]]
df = df.merge(markets, on="pm_market_id", how="left")

df


In [ ]:
# Cell 1 — setup
import os, time, json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import requests
from sentence_transformers import SentenceTransformer

BASE = os.getenv("POLYMARKET_API_BASE_URL") or "https://gamma-api.polymarket.com"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "BIT-CaseStudy/0.1"})

model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")  # or "cpu"


In [ ]:
import psycopg, os
conn = psycopg.connect(os.getenv("DATABASE_URL"))
with conn.cursor() as cur:
    cur.execute("select now(), current_database(), version();")
    print(cur.fetchone())
conn.close()


In [ ]:
import os
from google import genai

# REPLACE with your actual key for this one test
client = genai.Client(api_key="<REDACTED>")

try:
    response = client.models.generate_content(
        model="gemini-2.0-flash", 
        contents="hi"
    )
    print("Success! Quota is active.")
except Exception as e:
    print(f"Error details: {e}")

In [ ]:
tags_r = requests.get("https://gamma-api.polymarket.com/tags", params={"limit":200})
tags_r.json()

In [ ]:
# Cell 1 — setup
import os, time, json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import requests
from sentence_transformers import SentenceTransformer

BASE = os.getenv("POLYMARKET_API_BASE_URL") or "https://gamma-api.polymarket.com"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "BIT-CaseStudy/0.1"})

model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")  # or "cpu"


In [ ]:
# Cell 2 — Gamma helpers + "markets for tag" (via /events?tag_id=...)
def get_json(path: str, params: Optional[dict] = None, timeout_s: int = 30) -> Any:
    """Helper to retrieve json from gamma API

    Args:
        path (str): gamma api path
        params (Optional[dict], optional): _description_. Defaults to None.
        timeout_s (int, optional): _description_. Defaults to 30.

    Returns:
        Any: _description_
    """
    url = f"{BASE.rstrip('/')}{path}"
    r = SESSION.get(url, params=params or {}, timeout=timeout_s)
    r.raise_for_status()
    return r.json()

def try_get_json(path: str, params: Optional[dict] = None, timeout_s: int = 30) -> Tuple[bool, Any]:
    try:
        return True, get_json(path, params=params, timeout_s=timeout_s)
    except requests.HTTPError as e:
        return False, {"error": str(e), "status_code": getattr(e.response, "status_code", None), "path": path, "params": params}

def fetch_event_detail(event_id: Any) -> Optional[dict]:
    ok, data = try_get_json(f"/events/{event_id}", params=None)
    return data if ok and isinstance(data, dict) else None

def extract_markets_from_event(ev: dict) -> List[dict]:
    mkts = ev.get("markets")
    if isinstance(mkts, list) and mkts:
        return [m for m in mkts if isinstance(m, dict)]
    # sometimes events list responses omit markets; try detail fetch
    detail = fetch_event_detail(ev.get("id"))
    if detail and isinstance(detail.get("markets"), list):
        return [m for m in detail["markets"] if isinstance(m, dict)]
    return []

def fetch_markets_for_tag(
    tag_id: int,
    *,
    max_events_pages: int = 5,
    events_page_size: int = 50,
    markets_cap: int = 25,
    sleep_s: float = 0.05,
) -> List[dict]:
    markets_by_id: Dict[int, dict] = {}

    for page in range(max_events_pages):
        params = {
            "tag_id": str(tag_id),
            "closed": "false",
            "limit": str(events_page_size),
            "offset": str(page * events_page_size),
            # harmless if ignored; helps on some deployments
            "include_markets": "true",
        }
        ok, data = try_get_json("/events", params=params)
        if not ok:
            raise RuntimeError(f"Failed /events for tag_id={tag_id}: {data}")

        if not isinstance(data, list) or len(data) == 0:
            break

        for ev in data:
            if not isinstance(ev, dict):
                continue
            for m in extract_markets_from_event(ev):
                mid = m.get("id")
                if mid is None:
                    continue
                try:
                    mid_i = int(mid)
                except Exception:
                    continue
                if mid_i not in markets_by_id and (m.get("question") or m.get("title")):
                    markets_by_id[mid_i] = m
                    if len(markets_by_id) >= markets_cap:
                        return list(markets_by_id.values())

        time.sleep(sleep_s)

    return list(markets_by_id.values())

def build_tag_profile_text(tag: dict, markets: List[dict], *, max_questions: int = 20) -> str:
    label = (tag.get("label") or "").strip()
    slug = (tag.get("slug") or "").strip()
    tid = str(tag.get("id") or "").strip()

    questions = []
    for m in markets:
        q = (m.get("question") or m.get("title") or "").strip()
        if q:
            questions.append(q)
        if len(questions) >= max_questions:
            break

    # Profile = tag identity + concrete examples (what the tag retrieves)
    lines = [
        f"Polymarket tag profile.",
        f"tag_id: {tid}",
        f"label: {label}",
        f"slug: {slug}",
        "example_markets:",
        *[f"- {q}" for q in questions],
    ]
    return "\n".join(lines)


In [ ]:
# Cell 3 — build tag+markets embeddings + similarity to your domains/themes
# Provide your selected tags as list[dict] with keys: id,label,slug
# Example: selected_tags = df_allow[["id","label","slug"]].to_dict("records")




In [ ]:

# --- usage ---
# 1) define themes/domains you want to compare against
themes = [
    "AI & Data. AI models, LLMs, OpenAI, data platforms, AI regulation, GPUs.",
    "Compute & Semiconductors. GPUs, chips, foundries, semiconductor equipment, export controls, TSMC, ASML.",
    "Cloud & Software Infrastructure. AWS, Azure, GCP, enterprise SaaS spend, datacenters, observability, security.",
    "Consumer Internet & Digital Media. Social platforms, ads, e-commerce, app stores, antitrust.",
    "Fintech & Market Infrastructure. Payments rails, exchanges, brokerages, trading infrastructure.",
    "Digital Assets & Blockchain Infrastructure. Bitcoin, Ethereum, stablecoins, crypto ETFs, exchanges, regulation.",
    "Discount rate / Fed / inflation / rates.",
    "Regulation / antitrust / SEC / DOJ / FTC.",
    "Tariffs / export controls / sanctions / geopolitics.",
]
theme_emb = model.encode(themes, normalize_embeddings=True)



In [ ]:
# Cell 4 — optionally cap to top K tags for embedding (keeps it fast)
TOP_K = 10000  # adjust
df_emb = df.head(TOP_K).copy()
df_emb[["id","label","slug","n_hits","hits"]].head(20)

In [ ]:
# Cell 5 — embed tag texts
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")  # or "cpu"

texts = (df_emb["label"].fillna("") + " (" + df_emb["slug"].fillna("") + ")").tolist()
emb = model.encode(texts, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
emb.shape


In [ ]:
# Cell 6 — rank tags by similarity to a small set of themes (domain+channels)
themes = [
    "AI & Data: AI models, data platforms, AI regulation, OpenAI, GPUs",
    "Compute & Semiconductors: chips, GPUs, export controls, TSMC, ASML",
    "Cloud & Software Infrastructure: AWS Azure GCP, SaaS, enterprise software",
    "Consumer Internet & Digital Media: social, ads, e-commerce, platforms",
    "Fintech & Market Infrastructure: payments, exchanges, trading infra",
    "Digital Assets & Blockchain Infrastructure: bitcoin ethereum crypto ETFs stablecoins",
    "Discount rate / Fed / rates / inflation",
    "Regulation / antitrust / SEC / DOJ / FTC",
    "Tariffs / export controls / sanctions / geopolitics",
    "Demand shock / consumer spend / recession",
    "Supply shock / shortages / supply chain",
    "Hinge Health"
]

theme_emb = model.encode(themes, normalize_embeddings=True)

SIM_THRESHOLD = 0.3  # try 0.3–0.5

top_per_theme = {}
for j, theme in enumerate(themes):
    sims = S[:, j]
    idx = np.where(sims >= SIM_THRESHOLD)[0]
    idx = idx[np.argsort(-sims[idx])]  # sort remaining by similarity desc

    top_per_theme[theme] = (
        df_emb.loc[idx, ["id", "label", "slug", "n_hits"]]
        .assign(sim=sims[idx])
        .reset_index(drop=True)
    )

# optional: see how many tags survive per theme
{k: len(v) for k, v in top_per_theme.items()}


In [ ]:
# import tdqm as notebook_tdqm
from sentence_transformers import SentenceTransformer
import torch
import time
import requests
from typing import List
import numpy as np

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")

In [ ]:
from polyscanner.ingestion.tag_base import run_tag_selection, DEFAULT_THEMES

res = run_tag_selection(
    tag_source="relevant_events",
    min_seed_hits=1,
    seed_exclude={"rates"},
    max_tag_candidates=10000,
    min_markets_per_tag=5,
    similarity_threshold=0.1,
    top_k_per_theme=25,
    dedupe_to_best_theme=False,
)

res["similarity_stats"]
{k: len(v) for k, v in res["selected_by_theme"].items()}


In [ ]:
res["theme_keys"]

In [ ]:
res["selected_by_theme"][res["theme_keys"][0]]

In [ ]:
# Given: res = run_tag_selection(...)

import re
import pandas as pd

# Index helpers
profiles_by_id = {p.tag_id: p for p in res["profiles"]}          # TagProfile
selected_by_theme = res["selected_by_theme"]                     # dict[str, list[dict]]

def theme_table(theme: str, n: int = 25) -> pd.DataFrame:
    rows = selected_by_theme.get(theme, [])
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if "sim" in df.columns:
        df = df.sort_values("sim", ascending=False)
    cols = [c for c in ["tag_id", "label", "slug", "sim", "n_hits", "n_markets"] if c in df.columns]
    return df[cols].head(n).reset_index(drop=True)

def show_tag_profile(tag_id: int, max_lines: int = 40) -> None:
    p = profiles_by_id.get(int(tag_id))
    if not p:
        print(f"tag_id={tag_id} not found in profiles")
        return
    txt = p.profile_text.strip()
    lines = txt.splitlines()
    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... ({len(lines)-max_lines} more lines)")

def inspect_theme(theme: str, n: int = 15, show_profiles: int = 3) -> pd.DataFrame:
    df = theme_table(theme, n=n)
    print(f"\n=== {theme} ===")
    if df.empty:
        print("No selected tags.")
        return df
    display(df)
    for tag_id in df["tag_id"].head(show_profiles).tolist():
        print(f"\n--- tag_id={tag_id} profile preview ---")
        show_tag_profile(int(tag_id), max_lines=25)
    return df


In [ ]:
# Inspect all themes quickly (tables only)
for theme in res["theme_keys"]:
    df = theme_table(theme)
    print(theme, "->", len(df))
    if not df.empty:
        display(df)


In [ ]:
# Deep inspect one theme (table + top profiles)
inspect_theme(res["themes"][0], n=20, show_profiles=5)

In [ ]:
# Optional: find "almost selected" tags for a theme using the stored similarity stats
# (useful when a theme has 0 results and you want to see what's closest)
import numpy as np

themes = res["themes"]
theme_emb = res["theme_embeddings"]
tag_emb = res["tag_embeddings"]
S = tag_emb @ theme_emb.T

def top_similar_profiles(theme: str, n: int = 20) -> pd.DataFrame:
    j = themes.index(theme)
    sims = S[:, j]
    idx = np.argsort(-sims)[:n]
    rows = []
    for i in idx:
        p = res["profiles"][int(i)]
        rows.append({
            "tag_id": p.tag_id,
            "label": p.label,
            "slug": p.slug,
            "sim": float(sims[i]),
            "n_hits": p.n_hits,
            "n_markets": p.n_markets,
        })
    return pd.DataFrame(rows)

# Example:
display(top_similar_profiles("Cloud & Software Infrastructure. AWS, Azure, GCP, enterprise SaaS spend, datacenters, observability, security.", n=30))
